# 🎓 Kho ảnh minh họa cho giáo viên — v2

**Khác v1:** điểm tìm kiếm là **kết hợp** giữa CLIP (nhìn hình) và so khớp văn bản
(tên file + nhãn). Xếp hạng theo **z-score** thay vì điểm thô, nên ngưỡng không còn
phụ thuộc vào model hay loại ảnh.

**Vì sao đổi:** với ảnh scan sách giáo khoa dày chữ, CLIP cho mọi ảnh điểm gần như
nhau (chênh nhau 0.02) nên thứ tự gần như ngẫu nhiên. Tên file lại chứa đúng thông
tin cần tìm — v2 dùng cả hai.

**Chuẩn bị:** `mon_hoc_v3.json` + file `.zip` kho ảnh chia thư mục theo mã môn
(`toan/`, `vat_ly/`, `hoa_hoc/`, `sinh_hoc/`). Đặt tên file **có nghĩa, không dấu**:
`chuoi_thuc_an_01.jpg` tốt hơn `IMG_2841.jpg` rất nhiều.

In [ ]:
!pip -q install sentence-transformers faiss-cpu gradio

In [ ]:
import os
import json
import shutil
import zipfile
import unicodedata
from datetime import datetime

import numpy as np
import faiss
import torch
import gradio as gr
from PIL import Image
from google.colab import files
from sentence_transformers import SentenceTransformer

THU_MUC_ANH = "/content/bo_anh"
THU_MUC_DATA = "/content/data"
DUOI_ANH = (".jpg", ".jpeg", ".png", ".webp")

MODEL_ANH = "sentence-transformers/clip-ViT-B-32"          # encode ảnh + text tiếng Anh
MODEL_TEXT_VI = "sentence-transformers/clip-ViT-B-32-multilingual-v1"   # text tiếng Việt
MODEL_VAN_BAN = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # so khớp chữ

Z_MAC_DINH = 1.8      # ngưỡng z-score coi là "đạt"
ALPHA_MAC_DINH = 0.5  # tỉ trọng CLIP so với so khớp văn bản

os.makedirs(THU_MUC_DATA, exist_ok=True)


def bo_dau(chuoi):
    chuoi = chuoi.replace("đ", "d").replace("Đ", "D")
    chuoi = unicodedata.normalize("NFD", chuoi)
    chuoi = "".join(c for c in chuoi if unicodedata.category(c) != "Mn")
    return "".join(c if c.isalnum() else "_" for c in chuoi.lower().strip()).strip("_")


def nhan_file(cac_duoi, loi_nhac):
    print(loi_nhac)
    da_tai = files.upload()
    if not da_tai:
        raise ValueError("Bạn chưa chọn file nào.")
    ten_file = list(da_tai.keys())[0]
    if not ten_file.lower().endswith(cac_duoi):
        raise ValueError(f"File phải có đuôi {cac_duoi}, bạn đã chọn: {ten_file}")
    return ten_file


def chuan_hoa_z(vec):
    """Đưa vector điểm về z-score: 0 = trung bình kho, 2 = nổi bật rõ rệt."""
    do_lech = float(vec.std())
    if do_lech < 1e-8:
        return np.zeros_like(vec)
    return (vec - float(vec.mean())) / do_lech

## 1. Tải lên chương trình học

In [ ]:
file_chuong_trinh = nhan_file((".json",), "📚 Chọn file mon_hoc_v3.json:")

with open(file_chuong_trinh, encoding="utf-8") as f:
    CHUONG_TRINH = json.load(f)

for mon in CHUONG_TRINH:
    mon.setdefault("ma_mon", bo_dau(mon["ten_mon_hoc"]))
    for chuong in mon["chuong"]:
        chuong.setdefault("key", f"{mon['id']}-{chuong['id']}")
        for bai in chuong["bai_hoc"]:
            bai.setdefault("key", f"{mon['id']}-{chuong['id']}-{bai['id']}")
            bai.setdefault(
                "truy_van",
                f"{mon['ten_mon_hoc']} {chuong['ten_chuong']} {bai['ten_bai']}")
            bai.setdefault("truy_van_en", bai["truy_van"])
            bai.setdefault("co_the_minh_hoa", True)

TEN_MON = [m["ten_mon_hoc"] for m in CHUONG_TRINH]
MON_THEO_TEN = {m["ten_mon_hoc"]: m for m in CHUONG_TRINH}
BAI_THEO_KEY = {
    b["key"]: {"mon": m, "chuong": c, "bai": b}
    for m in CHUONG_TRINH for c in m["chuong"] for b in c["bai_hoc"]
}

print(f"✅ {len(CHUONG_TRINH)} môn, {len(BAI_THEO_KEY)} bài học.")

## 2. Giải nén kho ảnh

Tên file được dùng làm **mô tả văn bản** cho ảnh, nên đặt tên có nghĩa là cách rẻ nhất
để cải thiện kết quả. Muốn mô tả kỹ hơn, đặt thêm file `nhan.json` ở gốc file zip:
`{"sinh_hoc/chuoi_thuc_an_01.jpg": "chuỗi thức ăn trong hệ sinh thái đồng cỏ"}`

In [ ]:
file_zip = nhan_file((".zip",), "🖼️ Chọn file ZIP chứa kho ảnh:")

if os.path.exists(THU_MUC_ANH):
    shutil.rmtree(THU_MUC_ANH)
os.makedirs(THU_MUC_ANH)
with zipfile.ZipFile(file_zip, "r") as zf:
    zf.extractall(THU_MUC_ANH)

nhan_tay = {}
duong_dan_nhan = os.path.join(THU_MUC_ANH, "nhan.json")
if os.path.exists(duong_dan_nhan):
    with open(duong_dan_nhan, encoding="utf-8") as f:
        nhan_tay = json.load(f)
    print(f"📝 Đã đọc {len(nhan_tay)} nhãn mô tả tay.")

duong_dan_anh, mo_ta_anh, ma_mon_anh = [], [], []
nhom_anh = {}
so_anh_loi = 0

for thu_muc_goc, _, ten_cac_file in os.walk(THU_MUC_ANH):
    if "__MACOSX" in thu_muc_goc:
        continue
    for ten_file in sorted(ten_cac_file):
        if ten_file.startswith("._") or not ten_file.lower().endswith(DUOI_ANH):
            continue
        duong_dan = os.path.join(thu_muc_goc, ten_file)
        try:
            with Image.open(duong_dan) as anh:
                anh.verify()
        except Exception:
            so_anh_loi += 1
            print("⚠️ Không đọc được:", ten_file)
            continue

        tuong_doi = os.path.relpath(duong_dan, THU_MUC_ANH)
        phan = tuong_doi.split(os.sep)
        ma_mon = bo_dau(phan[0]) if len(phan) > 1 else "chung"

        # Mô tả văn bản: nhãn tay nếu có, không thì suy từ tên file
        if tuong_doi.replace(os.sep, "/") in nhan_tay:
            mo_ta = nhan_tay[tuong_doi.replace(os.sep, "/")]
        else:
            goc_ten = os.path.splitext(phan[-1])[0]
            tu = [t for t in goc_ten.replace("-", "_").split("_")
                  if t and not t.isdigit()]
            mo_ta = " ".join(tu)

        nhom_anh.setdefault(ma_mon, []).append(len(duong_dan_anh))
        duong_dan_anh.append(duong_dan)
        mo_ta_anh.append(mo_ta)
        ma_mon_anh.append(ma_mon)

if not duong_dan_anh:
    raise ValueError("Không tìm thấy ảnh hợp lệ nào trong file ZIP.")

print(f"✅ {len(duong_dan_anh)} ảnh hợp lệ, {so_anh_loi} ảnh lỗi.")
for ma, ds in sorted(nhom_anh.items()):
    print(f"   • {ma}: {len(ds)} ảnh")

vo_nghia = [os.path.basename(p) for p, m in zip(duong_dan_anh, mo_ta_anh)
            if len(m) < 4 or m.lower().startswith(("img", "dsc", "screenshot"))]
if vo_nghia:
    print(f"\n⚠️ {len(vo_nghia)} ảnh có tên không mang thông tin "
          f"(vd: {', '.join(vo_nghia[:3])}) — phần so khớp văn bản sẽ vô dụng với "
          f"những ảnh này. Nên đổi tên hoặc bổ sung nhan.json.")

## 3. Nạp mô hình

Ba mô hình, mỗi cái một việc:
- `clip-ViT-B-32` — mã hóa **ảnh**, và mã hóa **truy vấn tiếng Anh** (bộ mã hóa gốc,
  khớp tốt hơn hẳn bản đa ngữ)
- `clip-ViT-B-32-multilingual-v1` — mã hóa **truy vấn tiếng Việt**
- `paraphrase-multilingual-MiniLM-L12-v2` — so khớp **chữ với chữ** (truy vấn ↔ tên file)

In [ ]:
thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", thiet_bi)

model_anh = SentenceTransformer(MODEL_ANH, device=thiet_bi)
model_text_vi = SentenceTransformer(MODEL_TEXT_VI, device=thiet_bi)
model_van_ban = SentenceTransformer(MODEL_VAN_BAN, device=thiet_bi)

## 4. Mã hóa ảnh và mô tả

In [ ]:
KICH_THUOC_LO = 32
cac_lo = []
print("🔄 Mã hóa ảnh...")
for i in range(0, len(duong_dan_anh), KICH_THUOC_LO):
    lo_anh = []
    for duong_dan in duong_dan_anh[i:i + KICH_THUOC_LO]:
        with Image.open(duong_dan) as anh:
            lo_anh.append(anh.convert("RGB"))
    cac_lo.append(model_anh.encode(lo_anh, batch_size=KICH_THUOC_LO,
                                   convert_to_numpy=True, normalize_embeddings=True))
    del lo_anh
    print(f"   {min(i + KICH_THUOC_LO, len(duong_dan_anh))}/{len(duong_dan_anh)}")

VECTOR_ANH = np.vstack(cac_lo).astype("float32")
SO_CHIEU = VECTOR_ANH.shape[1]
del cac_lo

print("🔄 Mã hóa mô tả văn bản...")
VECTOR_MO_TA = model_van_ban.encode(
    mo_ta_anh, batch_size=64, convert_to_numpy=True,
    normalize_embeddings=True, show_progress_bar=True).astype("float32")

INDEX_TAT_CA = faiss.IndexFlatIP(SO_CHIEU)
INDEX_TAT_CA.add(VECTOR_ANH)

INDEX_THEO_MON = {}
anh_chung = nhom_anh.get("chung", [])
for mon in CHUONG_TRINH:
    vi_tri = nhom_anh.get(mon["ma_mon"], []) + anh_chung
    if not vi_tri:
        continue
    idx = faiss.IndexFlatIP(SO_CHIEU)
    idx.add(VECTOR_ANH[vi_tri])
    INDEX_THEO_MON[mon["ma_mon"]] = (idx, vi_tri)

np.save(os.path.join(THU_MUC_DATA, "anh_emb.npy"), VECTOR_ANH)
faiss.write_index(INDEX_TAT_CA, os.path.join(THU_MUC_DATA, "faiss.index"))
with open(os.path.join(THU_MUC_DATA, "anh_manifest.json"), "w", encoding="utf-8") as f:
    json.dump({
        "model_anh": MODEL_ANH,
        "model_van_ban": MODEL_VAN_BAN,
        "dim": int(SO_CHIEU),
        "so_luong": len(duong_dan_anh),
        "cap_nhat": datetime.now().isoformat(timespec="seconds"),
        "anh": [{"idx": i,
                 "id": os.path.splitext(os.path.basename(p))[0],
                 "file": os.path.relpath(p, THU_MUC_ANH),
                 "ma_mon": ma_mon_anh[i],
                 "mo_ta": mo_ta_anh[i]}
                for i, p in enumerate(duong_dan_anh)],
    }, f, ensure_ascii=False, indent=2)

assert VECTOR_ANH.shape[0] == INDEX_TAT_CA.ntotal == len(duong_dan_anh) == len(mo_ta_anh)
print(f"✅ {INDEX_TAT_CA.ntotal} ảnh, {SO_CHIEU} chiều, "
      f"{len(INDEX_THEO_MON)} kho riêng theo môn.")

## 5. Tìm kiếm kết hợp

`alpha = 1.0` là CLIP thuần (đúng như v1), `alpha = 0.0` là so khớp tên file thuần.
Với ảnh scan sách giáo khoa, `alpha ≈ 0.4–0.6` thường cho kết quả tốt nhất.

In [ ]:
def tim_kiem(cau_truy_van, ma_mon=None, so_luong=6,
             che_do="vi", alpha=ALPHA_MAC_DINH):
    if not cau_truy_van or not cau_truy_van.strip():
        return []
    cau_truy_van = cau_truy_van.strip()

    if ma_mon and ma_mon in INDEX_THEO_MON:
        index, ban_do = INDEX_THEO_MON[ma_mon]
    else:
        index, ban_do = INDEX_TAT_CA, list(range(len(duong_dan_anh)))
    ban_do = np.asarray(ban_do)

    # --- Phần 1: CLIP nhìn hình ---
    bo_ma_hoa = model_anh if che_do == "en" else model_text_vi
    vector_cau = bo_ma_hoa.encode([cau_truy_van], convert_to_numpy=True,
                                  normalize_embeddings=True).astype("float32")
    diem, vi_tri = index.search(vector_cau, index.ntotal)
    diem_clip = np.zeros(index.ntotal, dtype="float32")
    diem_clip[vi_tri[0]] = diem[0]

    # --- Phần 2: so khớp chữ với chữ ---
    vector_chu = model_van_ban.encode([cau_truy_van], convert_to_numpy=True,
                                      normalize_embeddings=True).astype("float32")
    diem_chu = VECTOR_MO_TA[ban_do] @ vector_chu[0]

    # --- Gộp bằng z-score để hai thang điểm so sánh được với nhau ---
    z_tong = alpha * chuan_hoa_z(diem_clip) + (1 - alpha) * chuan_hoa_z(diem_chu)

    so_luong = max(1, min(int(so_luong), len(ban_do)))
    thu_tu = np.argsort(-z_tong)[:so_luong]

    ket_qua = []
    for hang, vt in enumerate(thu_tu, start=1):
        goc = int(ban_do[vt])
        ket_qua.append({
            "hang": hang,
            "vi_tri": goc,
            "duong_dan": duong_dan_anh[goc],
            "ten_file": os.path.basename(duong_dan_anh[goc]),
            "mo_ta": mo_ta_anh[goc],
            "z": float(z_tong[vt]),
            "clip": float(diem_clip[vt]),
            "chu": float(diem_chu[vt]),
        })
    return ket_qua


def lam_gallery(ket_qua, nguong_z):
    return [(r["duong_dan"],
             f"{'✅' if r['z'] >= nguong_z else '⚠️'} #{r['hang']} {r['ten_file']} · "
             f"z={r['z']:.2f} · clip={r['clip']:.3f} · chữ={r['chu']:.3f}")
            for r in ket_qua]


def nhan_xet(ket_qua, nguong_z):
    if not ket_qua:
        return "Không có kết quả."
    dat = sum(1 for r in ket_qua if r["z"] >= nguong_z)
    dong = [f"Top-1: **{ket_qua[0]['ten_file']}** — z = **{ket_qua[0]['z']:.2f}** "
            f"(clip {ket_qua[0]['clip']:.3f}, chữ {ket_qua[0]['chu']:.3f}) · "
            f"{dat}/{len(ket_qua)} ảnh đạt ngưỡng."]
    if len(ket_qua) > 1:
        cach = ket_qua[0]["z"] - ket_qua[1]["z"]
        dong.append(f"Khoảng cách top-1 và top-2: **{cach:.2f}**")
        if cach < 0.15:
            dong.append("⚠️ Hai ảnh đầu gần như ngang nhau — hệ thống chưa thực sự "
                        "phân biệt được, nên chọn tay.")
    if ket_qua[0]["z"] < nguong_z:
        dong.append("⚠️ Không ảnh nào nổi bật so với phần còn lại của kho. "
                    "Thử viết câu mô tả cụ thể hơn, đổi sang tiếng Anh, "
                    "hoặc giảm alpha để dựa nhiều hơn vào tên file.")
    return "  \n".join(dong)


def lay_chuong(ten_mon):
    return [c["ten_chuong"] for c in MON_THEO_TEN[ten_mon]["chuong"]]


def lay_bai(ten_mon, ten_chuong):
    for chuong in MON_THEO_TEN[ten_mon]["chuong"]:
        if chuong["ten_chuong"] == ten_chuong:
            return [b["ten_bai"] for b in chuong["bai_hoc"]], chuong
    return [], None


def lay_bai_hoc(ten_mon, ten_chuong, ten_bai):
    _, chuong = lay_bai(ten_mon, ten_chuong)
    if chuong is None:
        return None
    for bai in chuong["bai_hoc"]:
        if bai["ten_bai"] == ten_bai:
            return bai
    return None


def truy_van_cua(bai, che_do):
    return bai["truy_van_en"] if che_do == "en" else bai["truy_van"]

## 6. Giao diện

In [ ]:
def canh_bao_bai(bai):
    if not bai.get("co_the_minh_hoa", True):
        return ("⚠️ Bài này được đánh dấu **khó minh họa bằng ảnh** — "
                "nên dùng ảnh mặc định của môn.")
    return ""


def doi_mon(ten_mon, che_do):
    ds_chuong = lay_chuong(ten_mon)
    ds_bai, _ = lay_bai(ten_mon, ds_chuong[0])
    bai = lay_bai_hoc(ten_mon, ds_chuong[0], ds_bai[0])
    return (gr.update(choices=ds_chuong, value=ds_chuong[0]),
            gr.update(choices=ds_bai, value=ds_bai[0]),
            truy_van_cua(bai, che_do), canh_bao_bai(bai))


def doi_chuong(ten_mon, ten_chuong, che_do):
    ds_bai, _ = lay_bai(ten_mon, ten_chuong)
    bai = lay_bai_hoc(ten_mon, ten_chuong, ds_bai[0])
    return (gr.update(choices=ds_bai, value=ds_bai[0]),
            truy_van_cua(bai, che_do), canh_bao_bai(bai))


def doi_bai(ten_mon, ten_chuong, ten_bai, che_do):
    bai = lay_bai_hoc(ten_mon, ten_chuong, ten_bai)
    if bai is None:
        return "", ""
    return truy_van_cua(bai, che_do), canh_bao_bai(bai)


def ma_che_do(nhan):
    return "en" if nhan.startswith("Tiếng Anh") else "vi"


def xu_ly_tim_theo_bai(ten_mon, cau, so_luong, nguong_z, trong_mon, nhan_che_do, alpha):
    ma_mon = MON_THEO_TEN[ten_mon]["ma_mon"] if trong_mon else None
    kq = tim_kiem(cau, ma_mon, so_luong, ma_che_do(nhan_che_do), alpha)
    return lam_gallery(kq, nguong_z), nhan_xet(kq, nguong_z)


def xu_ly_ca_chuong(ten_mon, ten_chuong, nguong_z, trong_mon, nhan_che_do, alpha):
    mon = MON_THEO_TEN[ten_mon]
    ma_mon = mon["ma_mon"] if trong_mon else None
    che_do = ma_che_do(nhan_che_do)
    _, chuong = lay_bai(ten_mon, ten_chuong)
    if chuong is None:
        return [], "Không tìm thấy chương."

    anh, dong = [], []
    for bai in chuong["bai_hoc"]:
        if not bai.get("co_the_minh_hoa", True):
            dong.append(f"| {bai['ten_bai']} | — | — | Khó minh họa |")
            continue
        kq = tim_kiem(truy_van_cua(bai, che_do), ma_mon, 2, che_do, alpha)
        if not kq:
            dong.append(f"| {bai['ten_bai']} | — | — | Không có kết quả |")
            continue
        tot = kq[0]
        dat = tot["z"] >= nguong_z
        anh.append((tot["duong_dan"],
                    f"{'✅' if dat else '⚠️'} {bai['ten_bai']} · z={tot['z']:.2f}"))
        cach = tot["z"] - kq[1]["z"] if len(kq) > 1 else 99
        ghi = "Đạt" if dat else "Chưa nổi bật"
        if cach < 0.15:
            ghi += ", sát ảnh thứ hai"
        dong.append(f"| {bai['ten_bai']} | {tot['ten_file']} | {tot['z']:.2f} | {ghi} |")

    bang = ("| Bài học | Ảnh đề xuất | z | Ghi chú |\n|---|---|---|---|\n"
            + "\n".join(dong))
    return anh, bang


def xu_ly_tim_tu_do(cau, ten_mon, so_luong, nguong_z, nhan_che_do, alpha):
    ma_mon = None if ten_mon == "Tất cả các môn" else MON_THEO_TEN[ten_mon]["ma_mon"]
    kq = tim_kiem(cau, ma_mon, so_luong, ma_che_do(nhan_che_do), alpha)
    return lam_gallery(kq, nguong_z), nhan_xet(kq, nguong_z)


def xu_ly_xuat_file(nguong_z, so_top, nhan_che_do, alpha):
    che_do = ma_che_do(nhan_che_do)
    tong_hop = {}
    for key, tt in BAI_THEO_KEY.items():
        mon, bai = tt["mon"], tt["bai"]
        if not bai.get("co_the_minh_hoa", True):
            tong_hop[key] = {"anh_id": None, "ly_do": "khong_the_minh_hoa",
                             "anh_thay_the": mon.get("anh_mac_dinh"), "da_duyet": False}
            continue
        kq = tim_kiem(truy_van_cua(bai, che_do), mon["ma_mon"], int(so_top), che_do, alpha)
        if not kq:
            tong_hop[key] = {"anh_id": None, "ly_do": "khong_co_ket_qua",
                             "da_duyet": False}
            continue
        tot = kq[0]
        top_k = [{"anh_id": r["ten_file"], "z": round(r["z"], 3),
                  "clip": round(r["clip"], 4)} for r in kq]
        ban_ghi = {"z": round(tot["z"], 3), "clip": round(tot["clip"], 4),
                   "top_k": top_k, "da_duyet": False}
        if tot["z"] >= nguong_z:
            ban_ghi.update({"anh_id": tot["ten_file"], "idx": tot["vi_tri"]})
        else:
            ban_ghi.update({"anh_id": None, "ly_do": "chua_noi_bat",
                            "anh_thay_the": mon.get("anh_mac_dinh")})
        tong_hop[key] = ban_ghi

    duong_dan = os.path.join(THU_MUC_DATA, "ket_qua_anh.json")
    with open(duong_dan, "w", encoding="utf-8") as f:
        json.dump({"che_do_truy_van": che_do, "alpha": alpha, "nguong_z": nguong_z,
                   "model_anh": MODEL_ANH, "model_van_ban": MODEL_VAN_BAN,
                   "chay_luc": datetime.now().isoformat(timespec="seconds"),
                   "ket_qua": tong_hop}, f, ensure_ascii=False, indent=2)

    so_dat = sum(1 for v in tong_hop.values() if v.get("anh_id"))
    return duong_dan, f"Đã xử lý **{len(tong_hop)}** bài, **{so_dat}** bài có ảnh đạt ngưỡng."


CHE_DO = ["Tiếng Việt (đa ngữ)", "Tiếng Anh (CLIP gốc, chính xác hơn)"]
mon_dau = TEN_MON[0]
chuong_dau = lay_chuong(mon_dau)[0]
ds_bai_dau, _ = lay_bai(mon_dau, chuong_dau)
bai_dau = lay_bai_hoc(mon_dau, chuong_dau, ds_bai_dau[0])


def khoi_gallery(nhan):
    return gr.Gallery(label=nhan, columns=3, height=620,
                      object_fit="contain", allow_preview=True)


with gr.Blocks(title="Kho ảnh minh họa cho giáo viên") as giao_dien:
    gr.Markdown("# 🎓 Kho ảnh minh họa cho giáo viên\n"
                "Điểm hiển thị là **z-score**: 0 = ngang trung bình kho, "
                "≥ 2 = nổi bật rõ rệt. Điểm CLIP thô chỉ để tham khảo.")

    with gr.Accordion("⚙️ Cách chấm điểm", open=False):
        rd_che_do = gr.Radio(CHE_DO, value=CHE_DO[0], label="Ngôn ngữ truy vấn")
        sl_alpha = gr.Slider(0.0, 1.0, value=ALPHA_MAC_DINH, step=0.05,
                             label="alpha — 1.0: chỉ nhìn hình (CLIP) · "
                                   "0.0: chỉ so tên file")
        sl_z = gr.Slider(0.0, 4.0, value=Z_MAC_DINH, step=0.1,
                         label="Ngưỡng z coi là đạt")

    with gr.Tab("📖 Tìm theo bài học"):
        with gr.Row():
            dd_mon = gr.Dropdown(TEN_MON, value=mon_dau, label="Môn học")
            dd_chuong = gr.Dropdown(lay_chuong(mon_dau), value=chuong_dau, label="Chương")
            dd_bai = gr.Dropdown(ds_bai_dau, value=ds_bai_dau[0], label="Bài học")
        tb_truy_van = gr.Textbox(value=bai_dau["truy_van"], lines=2,
                                 label="Câu mô tả dùng để tìm ảnh (sửa được)")
        md_canh_bao = gr.Markdown(canh_bao_bai(bai_dau))
        with gr.Row():
            sl_so_luong = gr.Slider(1, 12, value=6, step=1, label="Số ảnh trả về")
            cb_trong_mon = gr.Checkbox(True, label="Chỉ tìm trong kho ảnh của môn này")
        nut_tim = gr.Button("🔍 Tìm ảnh", variant="primary")
        md_ket_qua = gr.Markdown()
        gallery_bai = khoi_gallery("Ảnh gợi ý")

        dd_mon.change(doi_mon, [dd_mon, rd_che_do],
                      [dd_chuong, dd_bai, tb_truy_van, md_canh_bao])
        dd_chuong.change(doi_chuong, [dd_mon, dd_chuong, rd_che_do],
                         [dd_bai, tb_truy_van, md_canh_bao])
        dd_bai.change(doi_bai, [dd_mon, dd_chuong, dd_bai, rd_che_do],
                      [tb_truy_van, md_canh_bao])
        rd_che_do.change(doi_bai, [dd_mon, dd_chuong, dd_bai, rd_che_do],
                         [tb_truy_van, md_canh_bao])
        nut_tim.click(xu_ly_tim_theo_bai,
                      [dd_mon, tb_truy_van, sl_so_luong, sl_z, cb_trong_mon,
                       rd_che_do, sl_alpha],
                      [gallery_bai, md_ket_qua])

    with gr.Tab("📚 Cả chương"):
        gr.Markdown("Gợi ý một ảnh cho **từng bài** trong chương.")
        with gr.Row():
            dd_mon2 = gr.Dropdown(TEN_MON, value=mon_dau, label="Môn học")
            dd_chuong2 = gr.Dropdown(lay_chuong(mon_dau), value=chuong_dau, label="Chương")
        cb_trong_mon2 = gr.Checkbox(True, label="Chỉ tìm trong kho ảnh của môn này")
        nut_chuong = gr.Button("🔍 Tìm ảnh cho cả chương", variant="primary")
        gallery_chuong = khoi_gallery("Mỗi bài một ảnh")
        md_bang = gr.Markdown()

        dd_mon2.change(lambda m: gr.update(choices=lay_chuong(m), value=lay_chuong(m)[0]),
                       dd_mon2, dd_chuong2)
        nut_chuong.click(xu_ly_ca_chuong,
                         [dd_mon2, dd_chuong2, sl_z, cb_trong_mon2, rd_che_do, sl_alpha],
                         [gallery_chuong, md_bang])

    with gr.Tab("🔎 Tìm tự do"):
        tb_tu_do = gr.Textbox(label="Mô tả ảnh cần tìm", lines=2,
                              placeholder="Ví dụ: chuỗi thức ăn trong hệ sinh thái")
        with gr.Row():
            dd_loc_mon = gr.Dropdown(["Tất cả các môn"] + TEN_MON,
                                     value="Tất cả các môn", label="Giới hạn môn")
            sl_so_luong3 = gr.Slider(1, 20, value=9, step=1, label="Số ảnh trả về")
        nut_tu_do = gr.Button("🔍 Tìm ảnh", variant="primary")
        md_tu_do = gr.Markdown()
        gallery_tu_do = khoi_gallery("Kết quả")

        nut_tu_do.click(xu_ly_tim_tu_do,
                        [tb_tu_do, dd_loc_mon, sl_so_luong3, sl_z, rd_che_do, sl_alpha],
                        [gallery_tu_do, md_tu_do])

    with gr.Tab("💾 Xuất kết quả"):
        gr.Markdown("Chạy tìm kiếm cho **toàn bộ** bài học và xuất `ket_qua_anh.json`.")
        sl_top_k = gr.Slider(1, 10, value=3, step=1, label="Lưu top-k để đối chiếu")
        nut_xuat = gr.Button("💾 Xuất file JSON", variant="primary")
        md_xuat = gr.Markdown()
        file_xuat = gr.File(label="Tải file kết quả")

        nut_xuat.click(xu_ly_xuat_file, [sl_z, sl_top_k, rd_che_do, sl_alpha],
                       [file_xuat, md_xuat])

## 7. Chạy ứng dụng

In [ ]:
giao_dien.launch(share=True, debug=True)